# Looks To The Sun — interactive demo

Run the full **LTTS** transformer-interpretability GUI in your browser, with no local install.

This notebook clones the repo, installs the Python deps, builds the Vue frontend, starts the FastAPI server, and exposes it through a public tunnel. Then you load a small model (`SmolLM-135M`, ~270 MB, public) and explore its attention, logit-lens, activations, and more.

**Tip:** a GPU runtime is faster but optional — `SmolLM-135M` runs fine on CPU. To use a GPU: *Runtime → Change runtime type → T4 GPU*.

Run the cells top to bottom. Setup takes a few minutes (mostly the frontend build).

## 1. Clone the repositories


In [ ]:
!git clone --depth 1 https://github.com/seruva19/LTTS.git
# Analysis modules live in their own repository and are loaded from a
# sibling checkout, so clone it next to LTTS - without it there is
# nothing to attach to a layer.
!git clone --depth 1 https://github.com/seruva19/LTTS-modules.git
%cd LTTS


## 2. Install Python dependencies

Colab already ships a working PyTorch build, so we install everything **except** `torch`/`torchvision`/`torchaudio` (reinstalling the pinned CUDA build would be slow and can clash with Colab's driver).

In [ ]:
# Drop the torch lines + the pytorch extra index; keep the rest.
!grep -vE '^(--extra-index-url|torch|torchvision|torchaudio)' requirements.txt > /tmp/ltts_req.txt
!pip install -q -r /tmp/ltts_req.txt
# Cloudflare Quick Tunnels need no account or token. HTTP/2 is used because
# some hosted notebook networks block QUIC/UDP.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!/tmp/cloudflared --version
print('Python deps and cloudflared installed.')

## 3. Build the frontend

Compiles the Vue app into `dist/`, which the server serves on the same origin — so the GUI works through any tunnel or proxy without URL configuration.

In [ ]:
!npm install --no-audit --no-fund --loglevel=error
!npm run build-client
print('Frontend built into dist/.')

## 4. Start the server and expose it

Starts the FastAPI server, then opens a **Cloudflare Quick Tunnel**. It needs no account or token and forwards the WebSocket used by the live UI. The cell prints a temporary public link; open it in a new tab.

> The link is public while this cell runs — anyone who has it can reach the demo. It stays up until you run the teardown cell or stop the runtime.
>
> If tunnel startup fails, the exception includes the last cloudflared log lines instead of silently printing “no tunnel”.

In [ ]:
import os, subprocess, time, urllib.request
from scripts.utils.quick_tunnel import start_quick_tunnel

PORT = 8000
env = dict(os.environ)
env.update(
    LTTS_HOST='127.0.0.1',
    LTTS_PORT=str(PORT),
    LTTS_SKIP_MODULE_VENVS='1',  # use the base env; skip per-module venvs in the demo
)

server = subprocess.Popen(['python', 'ltts.py'], cwd=os.getcwd(), env=env)

# Wait for the server to come up (importing torch/transformers takes a moment).
health = f'http://127.0.0.1:{PORT}/'
for i in range(90):
    try:
        if urllib.request.urlopen(health, timeout=2).status == 200:
            print(f'Server is up after ~{i*2}s.')
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Server did not start - check the cell output above.')

# Confirm the modules repository was found - an empty list means the
# sibling checkout is missing and nothing can be attached to a layer.
import json as _json
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/ltts/modules', timeout=10) as _r:
    _modules = _json.load(_r).get('modules', {})
print(f'{len(_modules)} analysis modules loaded.')
if not _modules:
    print('No modules found - re-run cell 1 so LTTS-modules is cloned next to LTTS.')

# Cloudflare Quick Tunnel. The helper reads output on a background thread,
# enforces a real timeout, and reports startup logs on failure.
tunnel, public_url, tunnel_log = start_quick_tunnel(
    PORT, executable='/tmp/cloudflared', timeout=90
)
# Confirm that the public route reaches LTTS before presenting it.
for i in range(30):
    try:
        if urllib.request.urlopen(public_url, timeout=5).status == 200:
            break
    except Exception:
        time.sleep(2)
else:
    tunnel.terminate()
    raise RuntimeError(f'Tunnel URL was assigned but LTTS was not reachable: {public_url}')
print('\n  >>> Open the LTTS GUI here:', public_url, '\n')

## 5. How to test it

Open the public link from the previous cell, then in the GUI:

1. **Load a model** - top menu **Models -> Load model...**, then click **Load** on `HuggingFaceTB/SmolLM-135M`. The first load downloads ~270 MB from Hugging Face and then renders the model's architecture as a tree (~30-60 s). *(`Llama-3.2-1B` and `gemma-3-4b-it` are gated and need a Hugging Face token; `SmolLM-135M` and `Qwen3-4B` are public.)*
2. **Pick a layer** - scroll to a transformer block and click a layer card to expand it.
3. **Attach a module** - in **ADD POST-LAYER MODULE**, choose e.g. `attn_visualizer` (attention heatmap), `logit_lens`, or `gradient_importance`.
4. **Run** - type a prompt in **Model Input**, then click **Run** in the toolbar.
5. The module's visualization (e.g. a causal attention heatmap) renders in the layer's panel.

Model-level modules (e.g. activation steering) attach via **+ Add Model Module** at the top of the blueprint.

To try a different checkpoint, paste any Hugging Face model id into the **Add model** field in the model list. Only its `config.json` is fetched to work out how to load it, so a wrong id fails immediately rather than after a long download. Mind Colab's disk and RAM: stay small.

---
When you're done, stop the tunnel and server with the cell below (or just close the runtime).

In [ ]:
tunnel.terminate()
server.terminate()
print('Tunnel and server stopped.')